## 02 — Data Preprocessing


In [1]:
import os
import shutil
import h5py
import pyspark
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, lit, row_number, when
from pyspark.sql.types import StringType, DoubleType
import pandas as pd
from tqdm import tqdm

In [2]:
# Unified target schema. Order = output column order. Type = cast applied.
# artist_name is allowed to be NULL in the final output; everything else is required.
TARGET_SCHEMA = [
    ("track_id",         StringType()),
    ("artist_id",        StringType()),
    ("track_name",       StringType()),
    ("artist_name",      StringType()),
    ("duration_ms",      DoubleType()),
    ("danceability",     DoubleType()),
    ("loudness",         DoubleType()),
    ("energy",           DoubleType()),
    ("tempo",            DoubleType()),
    ("liveness",         DoubleType()),
    ("valence",          DoubleType()),
    ("speechiness",      DoubleType()),
    ("acousticness",     DoubleType()),
    ("instrumentalness", DoubleType()),
]

# Output directories
PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"

## The next 2 cells are for pyspark and hadoop to run on my device


In [3]:
#Note: The following environment variable setup is crucial for PySpark to work correctly on My development machine (Windows 11). If you encounter issues with PySpark, make sure these are set as shown below.

import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [4]:
#Note: The following Spark configuration is optimized for running PySpark on Windows in local mode. Adjust these settings if you run into performance issues or if your machine has different resources.


# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)



Spark version: 3.5.8


Helper functions


In [5]:
def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

In [6]:
def to_target(df, mapping):
    """Project df into TARGET_SCHEMA. mapping = {target_col: source_col}.
    Any target column not in mapping becomes NULL (cast to its target type)."""
    return df.select(*[
        (col(mapping[c]).cast(t) if c in mapping else lit(None).cast(t)).alias(c)
        for c, t in TARGET_SCHEMA
    ])


## PRIMARY SONGS DATASETS CLEANING


preprocessing needed for the dataset spotify_2023 to join for feature extraction


In [7]:
# DS2 spotify_2023 is split across multiple files. We only need albums (for
# track_id, artist_id, track_name, artist_0, duration_ms) joined to features
# (for the 9 audio features).
df2_albums = spark.read.csv(
    "../data/raw/spotify_2023/spotify-albums_data_2023.csv",
    header=True, inferSchema=True,
)
df2_features = spark.read.csv(
    "../data/raw/spotify_2023/spotify_features_data_2023.csv",
    header=True, inferSchema=True,
)

df2 = (
    df2_albums.select("track_id", "artist_id", "duration_ms", "track_name", "artist_0")
    .join(
        df2_features.select(
            col("id").alias("track_id"),
            "danceability", "loudness", "tempo", "energy",
            "liveness", "speechiness", "acousticness",
            "instrumentalness", "valence",
        ),
        on="track_id",
        how="left",
    )
)

df2.printSchema()
df2.show(5, truncate=False)

root
 |-- track_id: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_0: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- valence: double (nullable = true)

+----------------------+----------------------+-----------+---------------------------------------------------------------------------------------------------------+--------------------------+------------+--------+-------+------+--------+-----------+------------+----------------+-------+
|track_id              |artist_id             |duration_ms|track_name                                                         

Combine all datasets


In [8]:

# ---- Dataset 1: dataset_of_songs/genres_v2.csv ----
df1_raw = spark.read.csv(
    "../data/raw/dataset_of_songs/genres_v2.csv",
    header=True, inferSchema=True,
)
df1_std = to_target(df1_raw, {
    "track_id":         "id",
    "track_name":       "song_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "loudness":         "loudness",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})

# ---- Dataset 2: spotify_2023 (df2 already built above) ----
df2_std = to_target(df2, {
    "track_id":         "track_id",
    "artist_id":        "artist_id",
    "track_name":       "track_name",
    "artist_name":      "artist_0",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "loudness":         "loudness",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})

# ---- Dataset 3: spotify_audio_features (April 2019 + Nov 2018, same schema) ----
df3_raw = (
    spark.read.csv(
        "../data/raw/spotify_audio_features/SpotifyAudioFeaturesApril2019.csv",
        header=True, inferSchema=True)
    .unionByName(spark.read.csv(
        "../data/raw/spotify_audio_features/SpotifyAudioFeaturesNov2018.csv",
        header=True, inferSchema=True))
)
df3_std = to_target(df3_raw, {
    "track_id":         "track_id",
    "track_name":       "track_name",
    "artist_name":      "artist_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "loudness":         "loudness",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})

# ---- Dataset 4: spotify_tracks/dataset.csv ----
df4_raw = spark.read.csv(
    "../data/raw/spotify_tracks/dataset.csv",
    header=True, inferSchema=True,
)
df4_std = to_target(df4_raw, {
    "track_id":         "track_id",
    "artist_name":      "artists",
    "track_name":       "track_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "loudness":         "loudness",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})

# ---- Dataset 5: ultimate_spotify_tracks/SpotifyFeatures.csv ----
df5_raw = spark.read.csv(
    "../data/raw/ultimate_spotify_tracks/SpotifyFeatures.csv",
    header=True, inferSchema=True,
)
df5_std = to_target(df5_raw, {
    "track_id":         "track_id",
    "artist_name":      "artist_name",
    "track_name":       "track_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "loudness":         "loudness",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})

# ---- Union all + drop rows without track_id + dedupe by track_id ----
# When the same track_id appears in multiple datasets, keep the row with the
# most non-null fields (so DS2 with artist_id beats sparser rows).
score_cols = [c for c, _ in TARGET_SCHEMA if c != "track_id"]
nonnull_score = sum(when(col(c).isNotNull(), 1).otherwise(0) for c in score_cols)
w = Window.partitionBy("track_id").orderBy(col("_score").desc())

# Spotify track IDs are exactly 22 base62 chars. Some source rows are
# corrupted by CSV-quote misparsing (e.g. classical track names with embedded
# quotes shift every column over by one), producing garbage track_ids and
# nonsense values like duration_ms=7.0 (actually a leaked track_number).
# Filter these out at the source before union.
VALID_ID = r"^[0-9A-Za-z]{22}$"

all_rows = (df1_std
    .unionByName(df2_std)
    .unionByName(df3_std)
    .unionByName(df4_std)
    .unionByName(df5_std)
).cache()

total_rows  = all_rows.count()
clean_rows  = all_rows.filter(col("track_id").rlike(VALID_ID))
clean_count = clean_rows.count()
print(f"Total rows across all datasets: {total_rows:,}")
print(f"Dropped {total_rows - clean_count:,} rows with malformed track_id "
      f"({(total_rows - clean_count) / total_rows:.2%})")

unified = (clean_rows
    .withColumn("_score", nonnull_score)
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_score", "_rn")
)

print(f"Unified row count after dedupe: {unified.count():,}")
unified.printSchema()
unified.show(5, truncate=False)
all_rows.unpersist()

Total rows across all datasets: 1,075,038
Dropped 2,281 rows with malformed track_id (0.21%)
Unified row count after dedupe: 828,821
root
 |-- track_id: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)

+----------------------+----------------------+--------------------------------------+------------------------+-----------+------------+--------+------+-------+--------+-------+-----------+------------+----------------+
|track_id              |artist_id             |track_name  

DataFrame[track_id: string, artist_id: string, track_name: string, artist_name: string, duration_ms: double, danceability: double, loudness: double, energy: double, tempo: double, liveness: double, valence: double, speechiness: double, acousticness: double, instrumentalness: double]

In [ ]:
# Final cleanup: drop rows where ANY column except artist_name is NULL,
# then save the unified dataset. artist_name is allowed to be NULL.
FINAL_OUT = os.path.join(PROCESSED_DIR, "unified_tracks.csv")
required  = [c for c, _ in TARGET_SCHEMA if c != "artist_name"]

before = unified.count()
final  = unified.dropna(subset=required)
after  = final.count()
print(f"Dropped {before - after:,} rows missing required fields "
      f"({(before - after) / before:.2%})")
print(f"Final row count: {after:,}")

spark_write_csv(final, FINAL_OUT)

Dropped 391,969 rows missing required fields (47.29%)
Final row count: 436,852
